# 30 — Proof-of-Concept: Streaming Concept-Drift Detector (Paper 3)

**Dijalankan di SageMaker.** Membuktikan (murah, tanpa AWS baru) bahwa detektor
*drift* berbasis **jendela-geser Wasserstein $W_1$ + CUSUM** menyala di titik
transisi domain. Kita bentuk satu *stream* buatan berurutan: **CIC → UNS → AWS**,
lalu ukur skor drift $S_t$ (rata-rata $W_1$ 9 fitur SFM terhadap acuan latih).

**Alur:** muat 9-fitur CIC/UNS (dari `*_flows_v2.csv` hasil nb21, atau bangun dari
mentah) + AWS (S3 `unsw-far/results/`) → z-score gabungan → stream → sliding $W_1$
+ CUSUM → plot + JSON → upload S3 `unsw-far/drift/`.

> PoC ini tahap awal Paper 3 (`paper3-streaming.tex`). Bukan closed-loop penuh
> (belum ada micro-few-shot/guardrail); fokus: apakah detektor menandai transisi.

In [ ]:
import importlib, sys, subprocess
pkgmap={'sklearn':'scikit-learn'}
need=[m for m in ('scipy','pandas','numpy','matplotlib','boto3') if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*[pkgmap.get(m,m) for m in need]],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='drift_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; rng=np.random.default_rng(SEED)
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
print('=== SEL 1 (import & konfigurasi) SELESAI ===')

## 2. Muat 9-fitur CIC / UNS / AWS

In [ ]:
def read_feats(files):
    dfs=[]
    for f in files:
        try:
            d=pd.read_csv(f)
            if all(c in d.columns for c in CANON): dfs.append(d[CANON])
        except Exception as ex: print('  skip',f,ex)
    return pd.concat(dfs,ignore_index=True) if dfs else None
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h
    return []

# CIC & UNS: pakai cache _v2 dari notebook 21 bila ada (satuan sudah dikoreksi).
cic=read_feats(first(['cic_flows_v2.csv','../cic_flows_v2.csv','../notebooks/cic_flows_v2.csv']))
uns=read_feats(first(['uns_flows_v2.csv','../uns_flows_v2.csv','../notebooks/uns_flows_v2.csv']))

# AWS: unduh *_flows.csv dari S3
AWS_DIR='aws_flows'; os.makedirs(AWS_DIR,exist_ok=True)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION)
    for o in s3.list_objects_v2(Bucket=S3_BUCKET,Prefix=f'{S3_PREFIX}/results/').get('Contents',[]):
        k=o['Key']
        if k.endswith('_flows.csv'):
            dst=os.path.join(AWS_DIR,os.path.basename(k))
            if not os.path.exists(dst): s3.download_file(S3_BUCKET,k,dst)
except Exception as e: print('S3 dilewati:',e)
aws=read_feats(sorted(glob.glob(os.path.join(AWS_DIR,'*_flows.csv'))))

assert cic is not None and uns is not None and aws is not None, 'Butuh cic_flows_v2.csv, uns_flows_v2.csv (jalankan nb21 dulu) + CSV AWS.'
print('n:', {'CIC':len(cic),'UNS':len(uns),'AWS':len(aws)})
RESULTS['n_flow']={'CIC':int(len(cic)),'UNS':int(len(uns)),'AWS':int(len(aws))}
print('=== SEL 2 (muat CIC/UNS/AWS) SELESAI ===')

## 3. Bentuk stream CIC → UNS → AWS + acuan latih

In [ ]:
# Acuan (ref) = sampel CIC (domain latih awal). Skor drift diukur terhadap ref ini.
NREF=5000; NSEG=6000  # ukuran acuan & tiap segmen stream
def samp(df,n): return df.iloc[rng.permutation(len(df))[:min(n,len(df))]].reset_index(drop=True)
ref = samp(cic, NREF)
stream = pd.concat([samp(cic,NSEG), samp(uns,NSEG), samp(aws,min(NSEG,len(aws)))], ignore_index=True)
seg_bounds = [NSEG, 2*NSEG]  # indeks transisi CIC->UNS dan UNS->AWS
labels_seg = ['CIC']*NSEG + ['UNS']*NSEG + ['AWS']*min(NSEG,len(aws))

# z-score gabungan (fit pada ref+stream) agar W1 setara antar-fitur
allX=pd.concat([ref,stream],ignore_index=True); mu=allX.mean(); sd=allX.std().replace(0,1)
refz=((ref-mu)/sd)[CANON]; strz=((stream-mu)/sd)[CANON]
print('stream len',len(strz),'transisi di',seg_bounds)
print('=== SEL 3 (bentuk stream) SELESAI ===')

## 4. Sliding-window $W_1$ + CUSUM

In [ ]:
W=1000; STEP=250  # ukuran & langkah jendela geser
def drift_score(win_z):
    return float(np.mean([wasserstein_distance(win_z[c].values, refz[c].values) for c in CANON]))
centers=[]; scores=[]
for start in range(0, len(strz)-W+1, STEP):
    win=strz.iloc[start:start+W]
    centers.append(start+W//2); scores.append(drift_score(win))
scores=np.array(scores); centers=np.array(centers)

# Baseline & ambang dari fase awal (asumsi awal = in-distribution CIC)
base_idx=centers < seg_bounds[0]-W  # jendela penuh di segmen CIC
mu0=float(scores[base_idx].mean()); sd0=float(scores[base_idx].std()+1e-9)
thr=mu0+3*sd0  # ambang 3-sigma

# CUSUM pada skor (drift lambat-kumulatif)
k=0.5*sd0; cusum=np.zeros(len(scores)); c=0.0
for i,s in enumerate(scores):
    c=max(0.0, c+(s-mu0)-k); cusum[i]=c
h=5*sd0  # ambang CUSUM
alarms_w1=centers[scores>thr]
alarms_cusum=centers[cusum>h]
RESULTS['detector']={'W':W,'STEP':STEP,'mu0':round(mu0,4),'sd0':round(sd0,4),
  'thr_3sigma':round(thr,4),'cusum_h':round(h,4),
  'first_alarm_w1':int(alarms_w1[0]) if len(alarms_w1) else None,
  'first_alarm_cusum':int(alarms_cusum[0]) if len(alarms_cusum) else None,
  'transitions':seg_bounds}
print('mu0=%.3f sd0=%.3f thr=%.3f | alarm W1 pertama @%s | alarm CUSUM pertama @%s'%(
    mu0,sd0,thr, RESULTS['detector']['first_alarm_w1'], RESULTS['detector']['first_alarm_cusum']))
print('=== SEL 4 (sliding W1 + CUSUM) SELESAI ===')

## 5. Plot skor drift & titik transisi

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(9,6),sharex=True)
ax[0].plot(centers,scores,'-o',ms=3,color='#4C72B0',label='skor drift $S_t$ ($W_1$ rata-rata)')
ax[0].axhline(thr,ls='--',color='#C44E52',label='ambang 3$\\sigma$')
for b in seg_bounds: ax[0].axvline(b,ls=':',color='gray')
ax[0].text(seg_bounds[0]/2,ax[0].get_ylim()[1]*0.9,'CIC',ha='center')
ax[0].text((seg_bounds[0]+seg_bounds[1])/2,ax[0].get_ylim()[1]*0.9,'UNS',ha='center')
ax[0].text((seg_bounds[1]+len(strz))/2,ax[0].get_ylim()[1]*0.9,'AWS',ha='center')
ax[0].set_ylabel('$S_t$'); ax[0].set_title('Streaming drift: $W_1$ jendela-geser vs acuan CIC'); ax[0].legend(fontsize=8)
ax[1].plot(centers,cusum,'-',color='#DD8452',label='CUSUM'); ax[1].axhline(h,ls='--',color='#C44E52',label='ambang $h$')
for b in seg_bounds: ax[1].axvline(b,ls=':',color='gray')
ax[1].set_xlabel('indeks flow (waktu stream)'); ax[1].set_ylabel('CUSUM'); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig('drift_stream_score.png')
print('=== SEL 5 (plot) SELESAI ===')

## 6. Simpan + UPLOAD S3

In [ ]:
jp=os.path.join(OUTDIR,'drift_poc_results.json')
with open(jp,'w') as f: json.dump(RESULTS,f,indent=2)
print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/drift/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/drift/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 6 (simpan + upload) SELESAI ===')
print('SEMUA SELESAI. Beri tahu asisten -> unduh s3://%s/%s/drift/ untuk analisis.'%(S3_BUCKET,S3_PREFIX))